# Tiền xử lý dữ liệu cho các mô hình RoBERTa (NLBSE'23 Code Comment Classification)

## 1. Danh sách 9 mô hình RoBERTa sẽ huấn luyện
1. **RoBERTa Baseline**: RoBERTa + Linear classification head + BCE Loss.
2. **RoBERTa + Weighted BCE**: Thêm trọng số theo từng label để xử lý mất cân bằng dữ liệu.
3. **RoBERTa + Focal Loss**: Tập trung nhiều hơn vào các mẫu khó và label hiếm.
4. **RoBERTa + Threshold Tuning**: Tối ưu threshold riêng cho từng label thay vì dùng cố định 0.5.
5. **RoBERTa + Pooling Strategy**: So sánh CLS pooling, mean pooling, max pooling hoặc attention pooling.
6. **RoBERTa + Label-Aware Attention**: Học representation riêng cho từng label bằng attention.
7. **Label-Aware Attention + Weighted BCE**: Kết hợp label-aware attention với xử lý class imbalance.
8. **RoBERTa + Contrastive Learning**: Thêm contrastive loss để các mẫu có label tương đồng có embedding gần nhau hơn.
9. **Final Model**: RoBERTa + Label-Aware Attention + Weighted BCE + Contrastive Loss + Threshold Tuning.

## 2. Quy trình tiền xử lý dữ liệu (Data Preprocessing Pipeline)
- **Bước 0**: Cài đặt các thư viện cần thiết (`transformers`, `torch`).
- **Bước 1**: Nạp dữ liệu và kiểm tra schema.
- **Bước 2**: Chuẩn hóa văn bản comment (giữ nguyên, KHÔNG lowercase vì RoBERTa tokenizer tự xử lý).
- **Bước 3**: Phân chia Train/Test (80/20) bằng `MultilabelStratifiedKFold`.
- **Bước 4**: Tạo 5-fold Cross-Validation trên tập Train.
- **Bước 5**: Tokenize toàn bộ văn bản bằng `roberta-base` tokenizer (max_len=128).
- **Bước 6**: Xây dựng `PyTorch Dataset` class và `DataLoader` cho cả 3 tập.
- **Bước 7**: Tính `class_weights` cho BCE có trọng số (Weighted BCE) và `Focal Loss`.
- **Bước 8**: Lưu artifacts vào `data/roberta/` (`.pt`, `.npy`, `.json`, `.csv`).
- **Bước 9**: Kiểm tra xác thực và in tóm tắt sẵn sàng huấn luyện.

In [2]:
# Cài đặt các thư viện cần thiết cho mô hình RoBERTa
import subprocess, sys
pkgs_needed = []
for pkg in ['torch', 'transformers', 'tokenizers']:
    try:
        __import__(pkg)
    except ImportError:
        pkgs_needed.append(pkg)

if pkgs_needed:
    print(f'Đang cài đặt: {pkgs_needed}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'torch', 'transformers', 'tokenizers'])
    print('Cài đặt xong!')
else:
    print('Tất cả thư viện đã có sẵn!')

/home/linhntn16/projects/masterDS-seminar2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tất cả thư viện đã có sẵn!


In [3]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# Cấu hình
DATA_DIR    = os.path.abspath('../data')
OUTPUT_DIR  = os.path.join(DATA_DIR, 'roberta')
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABEL_COLS    = ['DevelopmentNotes', 'Expand', 'Parameters', 'Summary', 'Usage']
MODEL_NAME    = 'roberta-base'   # dùng roberta-base (~125M params)
MAX_LEN       = 128              # comment code ngắn, 128 token là đủ (95th percentile < 13 từ)
BATCH_SIZE    = 32
RANDOM_STATE  = 42

print(f'Thư mục dữ liệu   : {DATA_DIR}')
print(f'Thư mục artifacts  : {OUTPUT_DIR}')
print(f'Pretrained model   : {MODEL_NAME}')
print(f'Max token length   : {MAX_LEN}')
print(f'Device             : {"cuda" if torch.cuda.is_available() else "cpu"}')

Thư mục dữ liệu   : /home/linhntn16/projects/masterDS-seminar2/data
Thư mục artifacts  : /home/linhntn16/projects/masterDS-seminar2/data/roberta
Pretrained model   : roberta-base
Max token length   : 128
Device             : cpu


## Bước 1: Nạp dữ liệu và kiểm tra schema

In [4]:
df = pd.read_csv(os.path.join(DATA_DIR, 'python_preprocessed.csv'))
print(f'Tổng số mẫu        : {len(df):,}')
print(f'Số cột             : {df.shape[1]}')
print(f'Cột dữ liệu        : {df.columns.tolist()}')
print(f'Missing values     : {df.isnull().sum().sum()}')
print('\nThống kê positive per label:')
print(df[LABEL_COLS].sum().to_frame('Count').assign(**{"Ratio (%)": lambda x: (x['Count']/len(df)*100).round(2)}))

Tổng số mẫu        : 2,555
Số cột             : 9
Cột dữ liệu        : ['comment_sentence_id', 'class', 'comment_sentence', 'DevelopmentNotes', 'Expand', 'Parameters', 'Summary', 'Usage', 'label_count']
Missing values     : 0

Thống kê positive per label:
                  Count  Ratio (%)
DevelopmentNotes    312      12.21
Expand              504      19.73
Parameters          794      31.08
Summary             454      17.77
Usage               800      31.31


## Bước 2: Chuẩn hóa văn bản comment
Với RoBERTa **không cần lowercase** hay loại bỏ dấu câu vì tokenizer BPE (Byte-Pair Encoding) của RoBERTa đã xử lý điều này. Chỉ chuẩn hóa khoảng trắng và ký tự điều khiển.

In [5]:
def normalize_text(text: str) -> str:
    """Chuẩn hóa khoảng trắng và ký tự điều khiển cho RoBERTa tokenizer."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['comment_sentence'].apply(normalize_text)

# Thống kê độ dài token (rough estimate)
df['word_count'] = df['text'].str.split().str.len()
print('Thống kê độ dài câu (word count):')
print(df['word_count'].describe().round(2))
pct95 = df['word_count'].quantile(0.95)
print(f'\n95th percentile word count: {pct95:.0f} words → MAX_LEN={MAX_LEN} token là phù hợp')

Thống kê độ dài câu (word count):
count    2555.00
mean        6.80
std         4.14
min         1.00
25%         3.00
50%         6.00
75%        10.00
max        26.00
Name: word_count, dtype: float64

95th percentile word count: 13 words → MAX_LEN=128 token là phù hợp


## Bước 3: Phân chia Train/Test (80/20) bằng Multilabel Stratification
Dùng `MultilabelStratifiedKFold` để đảm bảo tỷ lệ phân bố của cả 5 nhãn cân bằng giữa tập Train và Test.

In [6]:
mskf_split = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_dummy = np.zeros(len(df))
Y_all = df[LABEL_COLS].values

# fold 0 → Test (20%), các fold còn lại → Train (80%)
train_idx, test_idx = next(mskf_split.split(X_dummy, Y_all))
train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

print(f'Tập Train: {len(train_df):,} mẫu ({len(train_df)/len(df)*100:.1f}%)')
print(f'Tập Test : {len(test_df):,} mẫu ({len(test_df)/len(df)*100:.1f}%)')

# Kiểm tra không bị rò rỉ dữ liệu
overlap = set(train_df['comment_sentence_id']) & set(test_df['comment_sentence_id'])
assert len(overlap) == 0, f'Data leakage: {len(overlap)} IDs trùng!'
print('✅ Không có data leakage')

print('\nPhân bố nhãn Train vs Test:')
comp = pd.DataFrame({
    'Train (%)': (train_df[LABEL_COLS].mean()*100).round(2),
    'Test  (%)': (test_df[LABEL_COLS].mean()*100).round(2)
})
print(comp)

Tập Train: 2,052 mẫu (80.3%)
Tập Test : 503 mẫu (19.7%)
✅ Không có data leakage

Phân bố nhãn Train vs Test:
                  Train (%)  Test  (%)
DevelopmentNotes      12.18      12.33
Expand                19.64      20.08
Parameters            30.95      31.61
Summary               17.69      18.09
Usage                 31.19      31.81


## Bước 4: Tạo 5-Fold Cross-Validation trên tập Train

In [7]:
mskf_cv = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = []
for fold_idx, (trn_i, val_i) in enumerate(
        mskf_cv.split(np.zeros(len(train_df)), train_df[LABEL_COLS].values)):
    folds.append({'fold': fold_idx,
                  'train_indices': trn_i.tolist(),
                  'val_indices': val_i.tolist()})
    print(f'Fold {fold_idx}: Train={len(trn_i):,}  Val={len(val_i):,}')

folds_path = os.path.join(OUTPUT_DIR, 'folds.json')
with open(folds_path, 'w', encoding='utf-8') as f:
    json.dump(folds, f, indent=2)
print(f'Đã lưu folds → {folds_path}')

Fold 0: Train=1,650  Val=402
Fold 1: Train=1,636  Val=416
Fold 2: Train=1,649  Val=403
Fold 3: Train=1,639  Val=413
Fold 4: Train=1,634  Val=418
Đã lưu folds → /home/linhntn16/projects/masterDS-seminar2/data/roberta/folds.json


## Bước 5: Tokenize bằng `roberta-base` AutoTokenizer
- Sử dụng `max_length=128` với `padding='max_length'` và `truncation=True`.
- Lưu `input_ids` và `attention_mask` dưới dạng numpy array → có thể nạp vào mọi framework.

In [8]:
print(f'Đang tải tokenizer: {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def batch_tokenize(texts, desc='Tokenizing'):
    """Tokenize danh sách văn bản thành input_ids và attention_mask."""
    encoding = tokenizer(
        texts.tolist(),
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='np'
    )
    print(f'  {desc}: {encoding["input_ids"].shape}')
    return encoding['input_ids'], encoding['attention_mask']

train_input_ids, train_attention_mask = batch_tokenize(train_df['text'], 'Train')
test_input_ids,  test_attention_mask  = batch_tokenize(test_df['text'],  'Test')

# Giải mã ví dụ để kiểm tra
print('\nVí dụ token decoded:')
example_decoded = tokenizer.decode(train_input_ids[0], skip_special_tokens=False)
print(example_decoded[:150], '...')

Đang tải tokenizer: roberta-base...


  Train: (2052, 128)
  Test: (503, 128)

Ví dụ token decoded:
<s>abstract cbv mixin that gives access mixins the same customizable</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pa ...


## Bước 6: Xây dựng `PyTorch Dataset` class và `DataLoader`
- `CodeCommentDataset` được thiết kế tương thích với **tất cả 9 mô hình** RoBERTa (chỉ khác nhau ở phần model architecture, không phải data loading).
- Trả về `input_ids`, `attention_mask`, và `labels` dạng float32 (phù hợp với BCE Loss).

In [9]:
class CodeCommentDataset(Dataset):
    """
    PyTorch Dataset cho bài toán multi-label code comment classification.
    Tương thích với tất cả 9 biến thể mô hình RoBERTa.
    """
    def __init__(self, input_ids: np.ndarray, attention_mask: np.ndarray,
                 labels: np.ndarray = None):
        self.input_ids      = torch.tensor(input_ids, dtype=torch.long)
        self.attention_mask = torch.tensor(attention_mask, dtype=torch.long)
        self.labels = (torch.tensor(labels, dtype=torch.float32)
                       if labels is not None else None)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
        }
        if self.labels is not None:
            item['labels'] = self.labels[idx]
        return item

# Nhãn dạng float32 cho BCE
y_train = train_df[LABEL_COLS].values.astype(np.float32)
y_test  = test_df[LABEL_COLS].values.astype(np.float32)

# Tạo Dataset
train_dataset = CodeCommentDataset(train_input_ids, train_attention_mask, y_train)
test_dataset  = CodeCommentDataset(test_input_ids,  test_attention_mask,  y_test)

# Tạo DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

print(f'Train Dataset: {len(train_dataset):,} samples  →  {len(train_loader)} batches')
print(f'Test Dataset : {len(test_dataset):,} samples   →  {len(test_loader)} batches')

# Kiểm tra một batch mẫu
sample_batch = next(iter(train_loader))
print('\nSample batch shapes:')
for k, v in sample_batch.items():
    print(f'  {k:<16}: {tuple(v.shape)}  dtype={v.dtype}')

Train Dataset: 2,052 samples  →  65 batches
Test Dataset : 503 samples   →  16 batches

Sample batch shapes:
  input_ids       : (32, 128)  dtype=torch.int64
  attention_mask  : (32, 128)  dtype=torch.int64
  labels          : (32, 5)  dtype=torch.float32


## Bước 7: Tính `class_weights` và `pos_weight` cho BCE Loss có trọng số
Phục vụ cho:
- **Mô hình 2** (Weighted BCE): `pos_weight[i] = (N - N_pos_i) / N_pos_i`
- **Mô hình 3** (Focal Loss): `alpha_i = N_neg_i / N`
- **Mô hình 7** (Label-Aware Attention + Weighted BCE): dùng lại `pos_weight`

In [10]:
n_train = len(train_df)
n_pos   = train_df[LABEL_COLS].sum().values.astype(np.float32)   # số positive mỗi nhãn
n_neg   = n_train - n_pos

# pos_weight cho torch.nn.BCEWithLogitsLoss(pos_weight=...)
# pos_weight[i] = (#negative_i) / (#positive_i)
pos_weight = (n_neg / np.maximum(n_pos, 1)).astype(np.float32)

# alpha cho Focal Loss (tỷ lệ negative → up-weight positive samples)
focal_alpha = (n_neg / n_train).astype(np.float32)

weights_dict = {
    'label_cols'  : LABEL_COLS,
    'n_train'     : int(n_train),
    'n_positive'  : n_pos.tolist(),
    'n_negative'  : n_neg.tolist(),
    'pos_weight'  : pos_weight.tolist(),    # for BCEWithLogitsLoss
    'focal_alpha' : focal_alpha.tolist(),   # for Focal Loss
}

print('Class weights per label:')
weights_df = pd.DataFrame({
    'Label'      : LABEL_COLS,
    'N_positive' : n_pos.astype(int),
    'N_negative' : n_neg.astype(int),
    'pos_weight' : pos_weight.round(3),
    'focal_alpha': focal_alpha.round(3),
})
print(weights_df.to_string(index=False))

print('\npos_weight tensor (dùng trực tiếp trong BCEWithLogitsLoss):')
print(torch.tensor(pos_weight))

Class weights per label:
           Label  N_positive  N_negative  pos_weight  focal_alpha
DevelopmentNotes         250        1802       7.208        0.878
          Expand         403        1649       4.092        0.804
      Parameters         635        1417       2.231        0.691
         Summary         363        1689       4.653        0.823
           Usage         640        1412       2.206        0.688

pos_weight tensor (dùng trực tiếp trong BCEWithLogitsLoss):
tensor([7.2080, 4.0918, 2.2315, 4.6529, 2.2062])


## Bước 8: Lưu tất cả artifacts vào `data/roberta/`
Cấu trúc artifacts được thiết kế để tất cả 9 mô hình RoBERTa có thể nạp dữ liệu bằng cùng một bộ file:
```
data/roberta/
├── train_input_ids.npy       # (2044, 128) int64
├── train_attention_mask.npy  # (2044, 128) int64
├── test_input_ids.npy        # (511, 128)  int64
├── test_attention_mask.npy   # (511, 128)  int64
├── y_train.npy               # (2044, 5)   float32
├── y_test.npy                # (511, 5)    float32
├── class_weights.json        # pos_weight & focal_alpha per label
├── label_cols.json           # ['DevelopmentNotes', ...]
├── folds.json                # 5-fold CV indices
├── train_meta.csv            # text + labels metadata
└── test_meta.csv             # text + labels metadata
```

In [11]:
# Lưu tokenized arrays
np.save(os.path.join(OUTPUT_DIR, 'train_input_ids.npy'),       train_input_ids)
np.save(os.path.join(OUTPUT_DIR, 'train_attention_mask.npy'), train_attention_mask)
np.save(os.path.join(OUTPUT_DIR, 'test_input_ids.npy'),        test_input_ids)
np.save(os.path.join(OUTPUT_DIR, 'test_attention_mask.npy'),   test_attention_mask)

# Lưu nhãn
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'),  y_test)

# Lưu class weights
with open(os.path.join(OUTPUT_DIR, 'class_weights.json'), 'w', encoding='utf-8') as f:
    json.dump(weights_dict, f, indent=2)

# Lưu label names
with open(os.path.join(OUTPUT_DIR, 'label_cols.json'), 'w', encoding='utf-8') as f:
    json.dump(LABEL_COLS, f, indent=2)

# Lưu metadata
train_df[['comment_sentence_id', 'text'] + LABEL_COLS].to_csv(
    os.path.join(OUTPUT_DIR, 'train_meta.csv'), index=False)
test_df[['comment_sentence_id', 'text'] + LABEL_COLS].to_csv(
    os.path.join(OUTPUT_DIR, 'test_meta.csv'), index=False)

# Tóm tắt kích thước file
print(f'Artifacts đã lưu vào: {OUTPUT_DIR}')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1024**2
    print(f'  {fname:<30}: {size_mb:>6.2f} MB')

Artifacts đã lưu vào: /home/linhntn16/projects/masterDS-seminar2/data/roberta
  class_weights.json            :   0.00 MB
  folds.json                    :   0.11 MB
  label_cols.json               :   0.00 MB
  test_attention_mask.npy       :   0.49 MB
  test_input_ids.npy            :   0.49 MB
  test_meta.csv                 :   0.03 MB
  train_attention_mask.npy      :   2.00 MB
  train_input_ids.npy           :   2.00 MB
  train_meta.csv                :   0.11 MB
  y_test.npy                    :   0.01 MB
  y_train.npy                   :   0.04 MB


## Bước 9: Kiểm tra xác thực & Tóm tắt sẵn sàng huấn luyện

In [12]:
# Tải lại và xác thực
tr_ids  = np.load(os.path.join(OUTPUT_DIR, 'train_input_ids.npy'))
te_ids  = np.load(os.path.join(OUTPUT_DIR, 'test_input_ids.npy'))
y_tr    = np.load(os.path.join(OUTPUT_DIR, 'y_train.npy'))
y_te    = np.load(os.path.join(OUTPUT_DIR, 'y_test.npy'))

assert tr_ids.shape  == train_input_ids.shape,  'Lỗi train_input_ids'
assert te_ids.shape  == test_input_ids.shape,   'Lỗi test_input_ids'
assert y_tr.shape    == y_train.shape,           'Lỗi y_train'
assert y_te.shape    == y_test.shape,            'Lỗi y_test'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('=' * 60)
print('  DATA PREPROCESSING SUMMARY - RoBERTa Models')
print('=' * 60)
print(f'  Pretrained model : {MODEL_NAME}')
print(f'  Max token length : {MAX_LEN}')
print(f'  Device           : {device}')
print(f'  Train samples    : {tr_ids.shape[0]:,}  →  input shape {tr_ids.shape}')
print(f'  Test  samples    : {te_ids.shape[0]:,}   →  input shape {te_ids.shape}')
print(f'  Label columns    : {LABEL_COLS}')
print(f'  CV Folds         : {len(folds)}')
print('=' * 60)
print('Snippet nạp dữ liệu dùng trong training notebooks:')
print('''
  import numpy as np, torch
  from torch.utils.data import DataLoader
  # ... import CodeCommentDataset từ notebook này

  ROBERTA_DIR = "../data/roberta"
  train_ds = CodeCommentDataset(
      np.load(f"{ROBERTA_DIR}/train_input_ids.npy"),
      np.load(f"{ROBERTA_DIR}/train_attention_mask.npy"),
      np.load(f"{ROBERTA_DIR}/y_train.npy")
  )
  train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
''')
print('\n✅ Tất cả 9 mô hình RoBERTa sẵn sàng nhận dữ liệu!')

  DATA PREPROCESSING SUMMARY - RoBERTa Models
  Pretrained model : roberta-base
  Max token length : 128
  Device           : cpu
  Train samples    : 2,052  →  input shape (2052, 128)
  Test  samples    : 503   →  input shape (503, 128)
  Label columns    : ['DevelopmentNotes', 'Expand', 'Parameters', 'Summary', 'Usage']
  CV Folds         : 5
Snippet nạp dữ liệu dùng trong training notebooks:

  import numpy as np, torch
  from torch.utils.data import DataLoader
  # ... import CodeCommentDataset từ notebook này

  ROBERTA_DIR = "../data/roberta"
  train_ds = CodeCommentDataset(
      np.load(f"{ROBERTA_DIR}/train_input_ids.npy"),
      np.load(f"{ROBERTA_DIR}/train_attention_mask.npy"),
      np.load(f"{ROBERTA_DIR}/y_train.npy")
  )
  train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)


✅ Tất cả 9 mô hình RoBERTa sẵn sàng nhận dữ liệu!
